# Day 11 — Treatment and Outcome Definition

## Project: EconoCausal — Dynamic Pricing via Double Machine Learning

### Objectives
- Identify potential treatment variables.
- Define the outcome variable.
- Identify possible pre-treatment covariates.
- Assess whether the available datasets support causal analysis.
- Document assumptions and limitations before model development.

In [1]:
import pandas as pd
import numpy as np

# Load datasets
train = pd.read_csv("../data/raw/train.csv")
transactions = pd.read_csv("../data/raw/customer_transaction_data.csv")
campaigns = pd.read_csv("../data/raw/campaign_data.csv")
customers = pd.read_csv("../data/raw/customer_demographics.csv")

# Display dataset dimensions
print("TRAIN DATASET:", train.shape)
print("TRANSACTION DATASET:", transactions.shape)
print("CAMPAIGN DATASET:", campaigns.shape)
print("CUSTOMER DATASET:", customers.shape)

TRAIN DATASET: (78369, 5)
TRANSACTION DATASET: (1324566, 7)
CAMPAIGN DATASET: (28, 4)
CUSTOMER DATASET: (760, 7)


In [2]:
# Inspect columns in each dataset

print("===== TRAIN DATASET =====")
print(train.columns.tolist())

print("\n===== TRANSACTION DATASET =====")
print(transactions.columns.tolist())

print("\n===== CAMPAIGN DATASET =====")
print(campaigns.columns.tolist())

print("\n===== CUSTOMER DATASET =====")
print(customers.columns.tolist())

===== TRAIN DATASET =====
['id', 'campaign_id', 'coupon_id', 'customer_id', 'redemption_status']

===== TRANSACTION DATASET =====
['date', 'customer_id', 'item_id', 'quantity', 'selling_price', 'other_discount', 'coupon_discount']

===== CAMPAIGN DATASET =====
['campaign_id', 'campaign_type', 'start_date', 'end_date']

===== CUSTOMER DATASET =====
['customer_id', 'age_range', 'marital_status', 'rented', 'family_size', 'no_of_children', 'income_bracket']


In [3]:
# Inspect potential outcome and treatment variables

print("===== OUTCOME VARIABLE =====")

if "redemption_status" in train.columns:
    print(train["redemption_status"].value_counts(dropna=False))
    print("\nRedemption rate:")
    print(f"{train['redemption_status'].mean() * 100:.2f}%")
else:
    print("redemption_status not found in training data.")

print("\n===== DISCOUNT-RELATED VARIABLES =====")

for df_name, df in [
    ("Train", train),
    ("Transactions", transactions),
    ("Campaigns", campaigns),
    ("Customers", customers)
]:
    discount_cols = [
        col for col in df.columns
        if "discount" in col.lower() or "price" in col.lower()
    ]

    print(f"\n{df_name}: {discount_cols}")

print("\n===== POTENTIAL TREATMENT-RELATED VARIABLES =====")

for df_name, df in [
    ("Train", train),
    ("Transactions", transactions),
    ("Campaigns", campaigns),
    ("Customers", customers)
]:
    treatment_cols = [
        col for col in df.columns
        if any(word in col.lower() for word in
               ["campaign", "coupon", "offer", "treatment", "redeem"])
    ]

    print(f"{df_name}: {treatment_cols}")

===== OUTCOME VARIABLE =====
redemption_status
0    77640
1      729
Name: count, dtype: int64

Redemption rate:
0.93%

===== DISCOUNT-RELATED VARIABLES =====

Train: []

Transactions: ['selling_price', 'other_discount', 'coupon_discount']

Campaigns: []

Customers: []

===== POTENTIAL TREATMENT-RELATED VARIABLES =====
Train: ['campaign_id', 'coupon_id']
Transactions: ['coupon_discount']
Campaigns: ['campaign_id', 'campaign_type']
Customers: []


In [4]:
import pandas as pd

# Preliminary treatment and outcome assessment
causal_variables = pd.DataFrame([
    {
        "variable_role": "Outcome (Y)",
        "variable_name": "redemption_status",
        "dataset": "train",
        "current_assessment": "Observed coupon redemption indicator",
        "status": "Candidate",
        "notes": "Measures redemption, not the causal effect of a discount."
    },
    {
        "variable_role": "Treatment (T)",
        "variable_name": "Not yet identified",
        "dataset": "To be determined",
        "current_assessment": "No verified treatment assignment",
        "status": "Pending validation",
        "notes": "Requires evidence of an assigned offer or discount."
    },
    {
        "variable_role": "Covariates (X)",
        "variable_name": "Customer demographics",
        "dataset": "customer_demographics",
        "current_assessment": "Potential pre-treatment characteristics",
        "status": "Candidate",
        "notes": "Confirm timing, relevance, and missingness before modeling."
    },
    {
        "variable_role": "Covariates (X)",
        "variable_name": "Campaign characteristics",
        "dataset": "campaign_data",
        "current_assessment": "Potential campaign-level characteristics",
        "status": "Candidate",
        "notes": "Verify campaign timing and how customers were assigned."
    },
    {
        "variable_role": "Identifiers",
        "variable_name": "customer_id, campaign_id, coupon_id",
        "dataset": "Multiple datasets",
        "current_assessment": "Keys for connecting records",
        "status": "Requires validation",
        "notes": "Identifiers alone do not establish treatment or exposure."
    }
])

display(causal_variables)

,variable_role,variable_name,dataset,current_assessment,status,notes
0,Outcome (Y),redemption_status,train,Observed coupon redemption indicator,Candidate,"Measures redemption, not the causal effect of ..."
1,Treatment (T),Not yet identified,To be determined,No verified treatment assignment,Pending validation,Requires evidence of an assigned offer or disc...
2,Covariates (X),Customer demographics,customer_demographics,Potential pre-treatment characteristics,Candidate,"Confirm timing, relevance, and missingness bef..."
3,Covariates (X),Campaign characteristics,campaign_data,Potential campaign-level characteristics,Candidate,Verify campaign timing and how customers were ...
4,Identifiers,"customer_id, campaign_id, coupon_id",Multiple datasets,Keys for connecting records,Requires validation,Identifiers alone do not establish treatment o...


In [5]:
# Save the treatment-outcome assessment
causal_variables.to_csv(
    "../reports/day11_treatment_outcome_assessment.csv",
    index=False
)

print("Day 11 assessment CSV saved successfully!")

Day 11 assessment CSV saved successfully!
